# Experiment: Expansion Lane Decision Gate

Objective:
- Rank near-term expansion lanes after the fixed first quote panel.
- Keep benchmark, hazard, and patient-treatment categories separate.
- Produce a small reproducible decision that can be copied into a finding and paper table.

Boundary:
- This is a computational evidence gate, not a wet-lab result.
- Scores are triage labels for Nakafa Lab research planning, not treatment advice.


In [1]:
from __future__ import annotations

from dataclasses import dataclass

SCORE_WEIGHTS = {
    "endpoint_fit": 0.25,
    "dual_readout_fit": 0.25,
    "human_thal_evidence": 0.20,
    "safety_margin": 0.15,
    "procurement_feasibility": 0.10,
    "assay_readiness": 0.05,
}

SCORE_WEIGHTS

{'endpoint_fit': 0.25, 'dual_readout_fit': 0.25, 'human_thal_evidence': 0.2, 'safety_margin': 0.15, 'procurement_feasibility': 0.1, 'assay_readiness': 0.05}

## Evidence Inputs

The lanes below are sourced from repo snapshots and findings created on 2026-04-27.
The gate favors items that can be tested in a dual HbF plus alpha-globin/autophagy assay without pretending they are patient candidates.


In [2]:
@dataclass(frozen=True)
class CandidateLane:
    """Represent one candidate lane for expansion-panel triage."""

    name: str
    role: str
    endpoint_fit: int
    human_thal_evidence: int
    dual_readout_fit: int
    safety_margin: int
    procurement_feasibility: int
    assay_readiness: int
    blocker: str
    boundary: str
    sources: tuple[str, ...]


def score_lane(lane: CandidateLane) -> float:
    """Return the weighted triage score for a candidate lane."""
    return round(
        lane.endpoint_fit * SCORE_WEIGHTS["endpoint_fit"]
        + lane.dual_readout_fit * SCORE_WEIGHTS["dual_readout_fit"]
        + lane.human_thal_evidence * SCORE_WEIGHTS["human_thal_evidence"]
        + lane.safety_margin * SCORE_WEIGHTS["safety_margin"]
        + lane.procurement_feasibility * SCORE_WEIGHTS["procurement_feasibility"]
        + lane.assay_readiness * SCORE_WEIGHTS["assay_readiness"],
        2,
    )


def gate_label(lane: CandidateLane) -> str:
    """Assign the operational label that should guide the next research action."""
    if lane.safety_margin <= 1:
        return "benchmark_only_high_caution"

    if lane.procurement_feasibility <= 1:
        return "conditional_or_benchmark_only"

    if lane.dual_readout_fit >= 4 and lane.assay_readiness >= 4:
        return "first_expansion_assay_probe"

    if lane.blocker:
        return "conditional_expansion"

    return "watchlist"


def ranked_lanes(lanes: tuple[CandidateLane, ...]) -> list[dict[str, object]]:
    """Return lane records sorted by score, then by name for stable output."""
    records = [
        {
            "lane": lane.name,
            "role": lane.role,
            "score": score_lane(lane),
            "gate": gate_label(lane),
            "blocker": lane.blocker or "none",
            "boundary": lane.boundary,
            "sources": lane.sources,
        }
        for lane in lanes
    ]
    return sorted(
        records, key=lambda record: (-float(record["score"]), str(record["lane"]))
    )

In [3]:
lanes = (
    CandidateLane(
        name="AMPKB1/NRF2/ULK1 autophagy probe",
        role="dual-readout pathway probe",
        endpoint_fit=4,
        human_thal_evidence=2,
        dual_readout_fit=5,
        safety_margin=3,
        procurement_feasibility=2,
        assay_readiness=4,
        blocker="defined research probe and lab assay required",
        boundary="promote as assay biology only, not as a patient candidate",
        sources=(
            "data/literature/pubmed/2026-04-27-expansion-ampkb1-nrf2-ulk1-hbf.json",
            "research/thalassemia/findings/2026-04-27-ampk-nrf2-expansion-gate.md",
            "research/thalassemia/findings/2026-04-27-ampkb1-autophagy-hbf-bridge.md",
        ),
    ),
    CandidateLane(
        name="Pyruvate kinase activator reference",
        role="clinical red-cell metabolism benchmark",
        endpoint_fit=4,
        human_thal_evidence=5,
        dual_readout_fit=2,
        safety_margin=4,
        procurement_feasibility=1,
        assay_readiness=2,
        blocker="access and legal procurement",
        boundary=(
            "use as endpoint benchmark unless a qualified lab can obtain "
            "a reference material"
        ),
        sources=(
            "research/thalassemia/findings/2026-04-27-pyruvate-kinase-red-cell-metabolism-benchmark.md",
        ),
    ),
    CandidateLane(
        name="T-BDMC-like curcuminoid analog",
        role="HbF chemistry expansion lane",
        endpoint_fit=4,
        human_thal_evidence=3,
        dual_readout_fit=2,
        safety_margin=2,
        procurement_feasibility=1,
        assay_readiness=3,
        blocker="compound identity, supplier, purity, and dose-window friction",
        boundary="conditional only; exact material and hemolysis gate must come first",
        sources=(
            "data/literature/pubmed/2026-04-27-expansion-t-bdmc-hbf-thalassemia.json",
            "research/thalassemia/findings/2026-04-27-t-bdmc-identity-resolution.md",
        ),
    ),
    CandidateLane(
        name="CRBN/WIZ/IKZF degrader class",
        role="BCL11A-adjacent HbF benchmark",
        endpoint_fit=4,
        human_thal_evidence=2,
        dual_readout_fit=3,
        safety_margin=1,
        procurement_feasibility=1,
        assay_readiness=2,
        blocker="drug-class safety, cytopenia, teratogenicity, and regulatory barrier",
        boundary=(
            "mechanistic comparator only; do not position as affordable treatment lead"
        ),
        sources=(
            "data/literature/pubmed/2026-04-27-expansion-crbn-ikzf-bcl11a-hbf.json",
            "data/literature/pubmed/2026-04-27-expansion-pomalidomide-hbf-thalassemia.json",
            "research/thalassemia/findings/2026-04-27-avadomide-crbn-hbf-evidence-boundary.md",
        ),
    ),
    CandidateLane(
        name="HBD/HbA2 delta-globin compensation",
        role="advanced globin-balance benchmark",
        endpoint_fit=4,
        human_thal_evidence=2,
        dual_readout_fit=2,
        safety_margin=2,
        procurement_feasibility=2,
        assay_readiness=2,
        blocker="specialized HBD/HbA2 readout and uncertain intervention chemistry",
        boundary="advanced benchmark, not first expansion material",
        sources=("research/thalassemia/findings/2026-04-27-delta-globin-hba2-gate.md",),
    ),
    CandidateLane(
        name="Hepcidin/ferroportin iron-axis modulation",
        role="disease-modifier benchmark",
        endpoint_fit=3,
        human_thal_evidence=3,
        dual_readout_fit=1,
        safety_margin=2,
        procurement_feasibility=1,
        assay_readiness=2,
        blocker="not a first HbF/globin-balance assay material",
        boundary=(
            "important for iron biology, not the first cure-discovery expansion lane"
        ),
        sources=(
            "research/thalassemia/findings/2026-04-27-iron-overload-organ-risk-gate.md",
        ),
    ),
)

ranked = ranked_lanes(lanes)
len(ranked)

6

## Results

The notebook turns the evidence map into an operational queue. A high score alone is not enough; the gate label controls whether an item is a first expansion probe, conditional expansion, or benchmark-only lane.


In [4]:
def markdown_table(records: list[dict[str, object]]) -> str:
    """Return a compact markdown table for the ranked lane records."""
    lines = [
        "| Rank | Lane | Score | Gate | Blocker |",
        "| --- | --- | ---: | --- | --- |",
    ]

    for rank, record in enumerate(records, start=1):
        lines.append(
            "| "
            + " | ".join(
                [
                    str(rank),
                    str(record["lane"]),
                    f"{float(record['score']):.2f}",
                    str(record["gate"]),
                    str(record["blocker"]),
                ]
            )
            + " |"
        )

    return "\n".join(lines)


print(markdown_table(ranked))

| Rank | Lane | Score | Gate | Blocker |
| --- | --- | ---: | --- | --- |
| 1 | AMPKB1/NRF2/ULK1 autophagy probe | 3.50 | first_expansion_assay_probe | defined research probe and lab assay required |
| 2 | Pyruvate kinase activator reference | 3.30 | conditional_or_benchmark_only | access and legal procurement |
| 3 | T-BDMC-like curcuminoid analog | 2.65 | conditional_or_benchmark_only | compound identity, supplier, purity, and dose-window friction |
| 4 | CRBN/WIZ/IKZF degrader class | 2.50 | benchmark_only_high_caution | drug-class safety, cytopenia, teratogenicity, and regulatory barrier |
| 5 | HBD/HbA2 delta-globin compensation | 2.50 | conditional_expansion | specialized HBD/HbA2 readout and uncertain intervention chemistry |
| 6 | Hepcidin/ferroportin iron-axis modulation | 2.10 | conditional_or_benchmark_only | not a first HbF/globin-balance assay material |


In [5]:
top_lane = ranked[0]
crbn_record = next(
    record for record in ranked if record["lane"] == "CRBN/WIZ/IKZF degrader class"
)
tbdmc_record = next(
    record for record in ranked if record["lane"] == "T-BDMC-like curcuminoid analog"
)

assert top_lane["lane"] == "AMPKB1/NRF2/ULK1 autophagy probe"
assert top_lane["gate"] == "first_expansion_assay_probe"
assert crbn_record["gate"] == "benchmark_only_high_caution"
assert tbdmc_record["gate"] == "conditional_or_benchmark_only"

{
    "first_expansion": top_lane["lane"],
    "first_expansion_boundary": top_lane["boundary"],
    "crbn_boundary": crbn_record["boundary"],
    "tbdmc_boundary": tbdmc_record["boundary"],
}

{'first_expansion': 'AMPKB1/NRF2/ULK1 autophagy probe', 'first_expansion_boundary': 'promote as assay biology only, not as a patient candidate', 'crbn_boundary': 'mechanistic comparator only; do not position as affordable treatment lead', 'tbdmc_boundary': 'conditional only; exact material and hemolysis gate must come first'}

## Decision

Promote `AMPKB1` / `NRF2` / `ULK1` / autophagy as the first expansion assay-probe lane if a qualified lab can source a defined research-grade probe and run dual HbF plus alpha-globin/autophagy readouts.

Keep T-BDMC-like curcuminoids conditional until exact material identity, supplier, purity, dose window, and hemolysis safety are solved.

Keep pyruvate kinase activation, CRBN/WIZ/IKZF degrader biology, HBD/HbA2 compensation, and hepcidin/ferroportin modulation as benchmarks or advanced lanes rather than first affordable-treatment claims.
